In [0]:
# ==============================================================================
# FASE 2: ENGENHARIA DE DADOS E LIMPEZA (CAMADA SILVER COM PYSPARK)
# ==============================================================================

from pyspark.sql.functions import col, upper, trim, to_timestamp

# 1. PROCESSAMENTO DA TABELA DE CLIENTES (CUSTOMERS)
df_bronze_customers = spark.read.table("workspace.default.bronze_customers")

df_silver_customers = (df_bronze_customers
    .dropna(subset=["customer_id"]) # Remove linhas sem ID de cliente
    .withColumn("customer_city", upper(trim(col("customer_city")))) # Padroniza cidade em maiúsculo
    .withColumn("customer_state", upper(trim(col("customer_state")))) # Padroniza estado
    .dropDuplicates(["customer_id"]) # Garante IDs únicos
)

# 2. PROCESSAMENTO DA TABELA DE PEDIDOS (ORDERS)
df_bronze_orders = spark.read.table("workspace.default.bronze_orders")

df_silver_orders = (df_bronze_orders
    .dropna(subset=["order_id"]) # Remove linhas sem ID de pedido
    # Converte as colunas de data de String para Timestamp real utilizando o Spark
    .withColumn("order_purchase_timestamp", to_timestamp(col("order_purchase_timestamp")))
    .withColumn("order_approved_at", to_timestamp(col("order_approved_at")))
    .withColumn("order_delivered_carrier_date", to_timestamp(col("order_delivered_carrier_date")))
    .withColumn("order_delivered_customer_date", to_timestamp(col("order_delivered_customer_date")))
    .withColumn("order_estimated_delivery_date", to_timestamp(col("order_estimated_delivery_date")))
    .dropDuplicates(["order_id"])
)

# 3. PROCESSAMENTO DA TABELA DE ITENS (ORDER ITEMS)
df_bronze_items = spark.read.table("workspace.default.bronze_order_items")

df_silver_items = (df_bronze_items
    .dropna(subset=["order_id", "order_item_id"])
    .withColumn("shipping_limit_date", to_timestamp(col("shipping_limit_date")))
    # Força os campos de valor a serem Double (numéricos decimais), caso o Spark não tenha inferido
    .withColumn("price", col("price").cast("double"))
    .withColumn("freight_value", col("freight_value").cast("double"))
)

# ==============================================================================
# PERSISTÊNCIA NA CAMADA SILVER (TABELAS DELTA LIMPAS)
# ==============================================================================

df_silver_customers.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_customers")
df_silver_orders.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_orders")
df_silver_items.write.format("delta").mode("overwrite").saveAsTable("workspace.default.silver_order_items")

print("Sucesso! O Spark aplicou as regras de limpeza e criou as tabelas na Camada Silver.")